# Preparación del entorno

In [ ]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import chi2_contingency, skew, kurtosis, mannwhitneyu, kruskal
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from collections import Counter
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
!pip install lightgbm
from lightgbm import LGBMClassifier
!pip install xgboost
from xgboost import XGBClassifier
!pip install catboost
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score, ConfusionMatrixDisplay

# Carga de datos

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

# Análisis Exploratorio de los Datos

In [ ]:
print(f"El conjunto de datos tiene {df.shape[0]} instancias y {df.shape[1]} variables:\n")
print(df.dtypes)

In [ ]:
df['HeartDiseaseorAttack'] = df['HeartDiseaseorAttack'].astype('category')
df['HighBP'] = df['HighBP'].astype('category')
df['HighChol'] = df['HighChol'].astype('category')
df['CholCheck'] = df['CholCheck'].astype('category')
df['Smoker'] = df['Smoker'].astype('category')
df['Stroke'] = df['Stroke'].astype('category')
df['Diabetes'] = df['Diabetes'].astype('category')
df['PhysActivity'] = df['PhysActivity'].astype('category')
df['Fruits'] = df['Fruits'].astype('category')
df['Veggies'] = df['Veggies'].astype('category')
df['HvyAlcoholConsump'] = df['HvyAlcoholConsump'].astype('category')
df['AnyHealthcare'] = df['AnyHealthcare'].astype('category')
df['NoDocbcCost'] = df['NoDocbcCost'].astype('category')
df['GenHlth'] = df['GenHlth'].astype('category')
df['DiffWalk'] = df['DiffWalk'].astype('category')
df['Sex'] = df['Sex'].astype('category')
df['Age'] = df['Age'].astype('category')
df['Education'] = df['Education'].astype('category')
df['Income'] = df['Income'].astype('category')

table = pd.DataFrame({
    "Tipo de variable": df.dtypes.values,
    "Valores faltantes (%)": df.isnull().mean().map("{:.2%}".format)
})
table.style.set_properties(**{'text-align': 'center'})

## Variable respuesta

In [ ]:
response = df['HeartDiseaseorAttack']
plt.figure(figsize=(8, 4))
relative_freq = response.value_counts(normalize=True, sort=False)
ax = sns.barplot(x=relative_freq.values, y=relative_freq.index, hue=relative_freq.index, palette='viridis', legend=False)
for i, v in enumerate(relative_freq.values):
    ax.text(v/2, i, f"{v:.2%}", va='center', ha='center', fontsize=10, color='black')
plt.title(f"Distribución de HeartDiseaseorAttack")
plt.xlabel("Frecuencia Relativa")
plt.ylabel("HeartDiseaseorAttack")
plt.show()

## Variables categóricas

In [ ]:
categorical_vars = df.select_dtypes(include=['category']).drop(columns=['HeartDiseaseorAttack']).columns
for var in categorical_vars:
    plt.figure(figsize=(8, 4))
    relative_freq = df[var].value_counts(normalize=True, sort=False)
    ax = sns.barplot(x=relative_freq.values, y=relative_freq.index, hue=relative_freq.index, palette='viridis', legend=False)
    ax.spines['right'].set_visible(False)
    for i, v in enumerate(relative_freq.values):
        ax.text(v, i, f"{v:.2%}", va='center', ha='left', fontsize=10, color='black')
    plt.title(f"Distribución de {var}")
    plt.xlabel("Frecuencia Relativa")
    plt.ylabel(var)
    plt.show()
    print("\n")

In [ ]:
def group_diabetes(x):
    if x == 0:
        return 0
    elif x in [1,2]:
        return 1

def group_genhlth(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3]:
        return 3
    elif x in [4,5]:
        return 4

def group_age(x):
    if x in [1,2,3,4]:
        return 1
    elif x in [5,6,7]:
        return 2
    elif x in [8,9,10]:
        return 3
    elif x in [11,12,13]:
        return 4

def group_education(x):
    if x in [1,2,3]:
        return 1
    elif x in [4]:
        return 2
    elif x in [5]:
        return 3
    elif x in [6]:
        return 4

def group_income(x):
    if x in [1, 2, 3]:
        return 1
    elif x in [4, 5]:
        return 2
    elif x in [6, 7]:
        return 3
    elif x in [8]:
        return 4

df['Diabetes'] = df['Diabetes'].apply(group_diabetes).astype('category')
df['GenHlth'] = df['GenHlth'].apply(group_genhlth).astype('category')
df['Age'] = df['Age'].apply(group_age).astype('category')
df['Education'] = df['Education'].apply(group_education).astype('category')
df['Income'] = df['Income'].apply(group_income).astype('category')

In [ ]:
new_categorical_vars = df[['Diabetes','GenHlth','Age','Education','Income']]
for var in new_categorical_vars:
    plt.figure(figsize=(8, 4))
    relative_freq = df[var].value_counts(normalize=True, sort=False)
    ax = sns.barplot(x=relative_freq.values, y=relative_freq.index, hue=relative_freq.index, palette='viridis', legend=False)
    ax.spines['right'].set_visible(False)
    for i, v in enumerate(relative_freq.values):
        ax.text(v, i, f"{v:.2%}", va='center', ha='left', fontsize=10, color='black')
    plt.title(f"Distribución de {var}")
    plt.xlabel("Frecuencia Relativa")
    plt.ylabel(var)
    plt.show()
    print("\n")

In [ ]:
results = []
heatmap_data = []
for var1, var2 in combinations(df.select_dtypes(include=['category']), 2):
    contingency_table = pd.crosstab(df[var1], df[var2])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    k = min(contingency_table.shape) - 1
    V = np.sqrt(chi2 / (n * k))
    results.append({
        "": var1,
        " ": var2,
        "Estadístico χ2": chi2,
        "p-valor": p_value,
        "V de Cramer": V
      })
    heatmap_data.append([var1, var2, V])
    heatmap_data.append([var2, var1, V])

results_df = pd.DataFrame(results).sort_values(by="V de Cramer", ascending=False)
results_df.style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_properties(subset=['Estadístico χ2','p-valor','V de Cramer'], **{'text-align': 'center'})

In [ ]:
df_heatmap = pd.DataFrame(heatmap_data, columns=["", " ", "V de Cramer"])
pivot_table = df_heatmap.pivot(index="", columns=" ", values="V de Cramer")
mask = np.triu(np.ones_like(pivot_table, dtype=bool))
plt.figure(figsize=(15, 10))
sns.heatmap(pivot_table, annot=True, cmap="Reds", linewidths=0.5, mask=mask, vmin = 0, vmax = 1)
plt.title("Heatmap de V de Cramer entre Variables Categóricas")
plt.show()

In [ ]:
df['Diet'] = df['Fruits'].astype('int64') + df['Veggies'].astype('int64')
df['Diet'] = df['Diet'].astype('category')

plt.figure(figsize=(8, 4))
relative_freq = df['Diet'].value_counts(normalize=True, sort=False)
ax = sns.barplot(x=relative_freq.values, y=relative_freq.index, hue=relative_freq.index, palette='viridis', legend=False)
ax.spines['right'].set_visible(False)
for i, v in enumerate(relative_freq.values):
  ax.text(v, i, f"{v:.2%}", va='center', ha='left', fontsize=10, color='black')
plt.title(f"Distribución de Diet")
plt.xlabel("Frecuencia Relativa")
plt.ylabel('Diet')
plt.show()
print("\n")

results = []
contingency_table = pd.crosstab(df['Diet'], df['HeartDiseaseorAttack'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
n = contingency_table.sum().sum()
k = min(contingency_table.shape) - 1
V = np.sqrt(chi2 / (n * k))
results.append({
  "": 'Diet',
  " ": 'HeartDiseaseorAttack',
  "Estadístico χ2": chi2,
  "p-valor": p_value,
  "V de Cramer": V
})
results_df = pd.DataFrame(results).sort_values(by="V de Cramer", ascending=False)
results_df.style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_properties(subset=['Estadístico χ2','p-valor','V de Cramer'], **{'text-align': 'center'})

In [ ]:
df = df.drop(columns=['CholCheck', 'AnyHealthcare', 'NoDocbcCost', 'HvyAlcoholConsump','Fruits','Veggies','Diet','DiffWalk'])

## Variables numéricas

In [ ]:
numerical_vars = df.select_dtypes(include=['int64'])
description = numerical_vars.describe().drop('count')
skew = numerical_vars.apply(skew)
description.loc['skew'] = skew
kurtosis = numerical_vars.apply(kurtosis)
description.loc['kurtosis'] = kurtosis
description.style.set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])

In [ ]:
for column in numerical_vars.columns:
  plt.figure(figsize=(7, 5))
  sns.histplot(numerical_vars[column], bins=20, kde=True)
  plt.title(f'Histograma de {column}')
  plt.xlabel('')
  plt.ylabel('')
  plt.show()
  print("\n")

In [ ]:
for column in numerical_vars.columns:
  plt.figure(figsize=(7, 5))
  sns.boxplot(x=numerical_vars[column])
  plt.title(f'Boxplot de {column}')
  plt.xlabel('')
  plt.show()
  print("\n")

In [ ]:
corr_matrix = numerical_vars.corr('spearman')
plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlación")
plt.show()

In [ ]:
results = []
for col in numerical_vars.columns:
    group0 = df[col][df['HeartDiseaseorAttack'] == 0]
    group1 = df[col][df['HeartDiseaseorAttack'] == 1]
    u_statistic, p_value_mw = mannwhitneyu(group0, group1, alternative='two-sided')
    n1 = len(group0)
    n2 = len(group1)
    mean_u = n1 * n2 / 2
    std_u = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    z = (u_statistic - mean_u) / std_u
    r = abs(z) / np.sqrt(n1 + n2)
    results.append({
        "Variable": col,
        "Estadístico Mann-Whitney": u_statistic,
        "p-valor": p_value_mw,
        "r de Rosenthal": r
    })

results_df = pd.DataFrame(results).sort_values(by="r de Rosenthal", ascending=False)
results_df.style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_properties(subset=['Estadístico Mann-Whitney', 'p-valor', 'r de Rosenthal'], **{'text-align': 'center'})

In [ ]:
results = []

groups_phys = [group["PhysHlth"].values for name, group in df.groupby("GenHlth", observed=True)]
stat_phys, p_phys = kruskal(*groups_phys)
n_phys = df["PhysHlth"].notnull().sum()
k_phys = len(groups_phys)
epsilon_sq_phys = (stat_phys - k_phys + 1) / (n_phys - k_phys)
results.append({
    "": 'PhysHlth-GenHlth',
    "Estadístico H": stat_phys,
    "p-valor": p_phys,
    "Epsilon²": epsilon_sq_phys
})

results_df = pd.DataFrame(results).sort_values(by="Epsilon²", ascending=False)
results_df

In [ ]:
df = df.drop(columns=['PhysHlth', 'MentHlth'])

# Construcción y evaluación de modelos basados únicamente en variables de autoevaluación

In [ ]:
df_autoevaluation = df.drop(columns=['HighBP', 'HighChol', 'Stroke', 'Diabetes', 'BMI'])
X_autoevaluation = df_autoevaluation.drop('HeartDiseaseorAttack', axis=1)
y_autoevaluation = df_autoevaluation['HeartDiseaseorAttack']
X_autoevaluation_train, X_autoevaluation_test, y_autoevaluation_train, y_autoevaluation_test = train_test_split(X_autoevaluation, y_autoevaluation, test_size=0.20, random_state=100473899, shuffle=True, stratify=y_autoevaluation)

## Random Forest

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [RandomForestClassifier()],
    'model__max_depth': [None, 10, 15, 20, 30],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__max_features': ['sqrt', 'log2', X_autoevaluation_train.shape[1] // 3]
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_autoevaluation_train, y_autoevaluation_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__max_features':}: {random_search.best_params_['model__max_features']}")

best_autoevaluation_model = random_search.best_estimator_
y_autoevaluation_pred = best_autoevaluation_model.predict(X_autoevaluation_test)
recall_test = recall_score(y_autoevaluation_test, y_autoevaluation_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_autoevaluation_test, y_autoevaluation_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Extreme Gradient Boosting

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', XGBClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [XGBClassifier(enable_categorical=True, use_label_encoder=False, eval_metric='logloss')],
    'model__max_depth': [3, 5, 7, 10],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05)
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_autoevaluation_train, y_autoevaluation_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")

best_autoevaluation_model = random_search.best_estimator_
y_autoevaluation_pred = best_autoevaluation_model.predict(X_autoevaluation_test)
recall_test = recall_score(y_autoevaluation_test, y_autoevaluation_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_autoevaluation_test, y_autoevaluation_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Light Boosting

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', LGBMClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [LGBMClassifier()],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05),
    'model__num_leaves': [15, 31, 63, 127, 255]
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_autoevaluation_train, y_autoevaluation_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")
print("\n",f"   - {'model__num_leaves':}: {random_search.best_params_['model__num_leaves']}")

best_autoevaluation_model = random_search.best_estimator_
y_autoevaluation_pred = best_autoevaluation_model.predict(X_autoevaluation_test)
recall_test = recall_score(y_autoevaluation_test, y_autoevaluation_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_autoevaluation_test, y_autoevaluation_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Categorical Boosting

In [ ]:
categorical_autoevaluation_list = X_autoevaluation_train.select_dtypes(include=['category']).columns.tolist()

pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', CatBoostClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [CatBoostClassifier(verbose=0, cat_features=categorical_autoevaluation_list)],
    'model__max_depth': [3, 5, 7, 10],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05)
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_autoevaluation_train, y_autoevaluation_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")

best_autoevaluation_model = random_search.best_estimator_
y_autoevaluation_pred = best_autoevaluation_model.predict(X_autoevaluation_test)
recall_test = recall_score(y_autoevaluation_test, y_autoevaluation_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_autoevaluation_test, y_autoevaluation_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

# Construcción y evaluación de modelos que incorporan las variables clínicas

In [ ]:
X = df.drop('HeartDiseaseorAttack', axis=1)
y = df['HeartDiseaseorAttack']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=100473899, shuffle=True, stratify=y)

## Random Forest

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [RandomForestClassifier()],
    'model__max_depth': [None, 10, 15, 20, 30],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__max_features': ['sqrt', 'log2', X_train.shape[1] // 3]
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_train, y_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__max_features':}: {random_search.best_params_['model__max_features']}")

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
recall_test = recall_score(y_test, y_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Extreme Gradient Boosting

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', XGBClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [XGBClassifier(enable_categorical=True, use_label_encoder=False, eval_metric='logloss')],
    'model__max_depth': [3, 5, 7, 10],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05)
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_train, y_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
recall_test = recall_score(y_test, y_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Light Boosting

In [ ]:
pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', LGBMClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [LGBMClassifier()],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05),
    'model__num_leaves': [15, 31, 63, 127, 255]
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_train, y_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")
print("\n",f"   - {'model__num_leaves':}: {random_search.best_params_['model__num_leaves']}")

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
recall_test = recall_score(y_test, y_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()

## Categorical Boosting

In [ ]:
categorical_list = X_train.select_dtypes(include=['category']).columns.tolist()

pipeline = Pipeline(steps=[
    ('sampler', SMOTE()),
    ('model', CatBoostClassifier())
])

param_grid = {
    'sampler': [SMOTE(), NearMiss()],
    'sampler__sampling_strategy': ['auto', 0.7, 0.8],
    'model': [CatBoostClassifier(verbose=0, cat_features=categorical_list)],
    'model__max_depth': [3, 5, 7, 10],
    'model__n_estimators': np.arange(100, 1001, 50),
    'model__learning_rate': np.arange(0.01, 0.31, 0.05)
}

kf = KFold(n_splits=5, shuffle=True, random_state=100473899)
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=20, cv=kf, scoring='recall', verbose=1, random_state=100473899, n_jobs=-1)
random_search.fit(X_train, y_train)

print("\n",f"La mejor sensibilidad es {random_search.best_score_:.4f} y es obtenida por la siguiente combinación de hiperparámetros:")
print("\n",f"- sampler: {type(random_search.best_params_['sampler']).__name__}")
print("\n",f"   - {'sampler__sampling_strategy':}: {random_search.best_params_['sampler__sampling_strategy']}")
print("\n",f"- {'model':}: {random_search.best_params_['model']}")
print("\n",f"   - {'model__max_depth':}: {random_search.best_params_['model__max_depth']}")
print("\n",f"   - {'model__n_estimators':}: {random_search.best_params_['model__n_estimators']}")
print("\n",f"   - {'model__learning_rate':}: {random_search.best_params_['model__learning_rate']}")

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
recall_test = recall_score(y_test, y_pred)
print("\n",f"Sensibilidad en el conjunto de validación: {recall_test:.4f}","\n")

cm_test = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_test).plot(ax=ax)
ax.set_title("Matriz de Confusión - Test")
plt.tight_layout()
plt.show()